# Exploratory statistical comparison at the largest training sizes

This notebook reads saved results; it does not retrain models or change the input files. It compares the largest evaluated training-plus-validation size for each task. Each metric is analyzed separately, using the saved bootstrap **mean** as the task score. Results are exploratory because there are only four cohort groups, one model fit per task and size, and the selected tasks do not constitute a random sample of clinical problems.

**Shared-subject tasks:** Heart and Diabetes-H are different prediction objectives from related subjects; Obesity and Obesity-Bin are different objectives from the same dataset. They are kept separate when ranking models within tasks. For cohort summaries and tests, each pair receives equal weight within one group. Grouping limits double-counting; it does not prove that the four resulting groups are independent.

For reproducibility the notebook requires complete results for the selected models, tasks, and metrics, and checks that models for a task used the same stored training indices, test indices, and test labels. Only trusted pickle files should be loaded.

In [ ]:
from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from statsmodels.stats.multitest import multipletests

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'results_metrics').is_dir() and (PROJECT_ROOT.parent / 'results_metrics').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS_DIR = PROJECT_ROOT / 'results_metrics' / 'current'

TASK_SIZES = {
    'heart': 10000,
    'diabetes_h': 10000,
    'diabetes_130': 10000,
    'obesity': 1688,
    'obesity_bin': 1688,
    'breast_cancer': 455,
}
COHORT_GROUPS = {
    'Heart + Diabetes-H': ['heart', 'diabetes_h'],
    'Diabetes-130': ['diabetes_130'],
    'Obesity + Obesity-Bin': ['obesity', 'obesity_bin'],
    'Breast Cancer': ['breast_cancer'],
}
MODELS = ['mlp', 'lr', 'rf', 'nam', 'kan', 'kan_gam', 'ebm', 'xgboost', 'lightgbm']
MODEL_LABELS = {
    'mlp': 'MLP', 'lr': 'LR', 'rf': 'RF', 'nam': 'NAM',
    'kan': 'Logistic-KAN', 'kan_gam': 'KAAM', 'ebm': 'EBM',
    'xgboost': 'XGBoost', 'lightgbm': 'LightGBM',
}
# These are the original main-table metrics. True means higher is better.
METRICS = {
    'accuracy': {'label': 'Accuracy', 'higher_is_better': True},
    'roc_auc': {'label': 'ROC-AUC', 'higher_is_better': True},
    'f1': {'label': 'F1', 'higher_is_better': True},
    'precision': {'label': 'Precision', 'higher_is_better': True},
    'recall': {'label': 'Recall', 'higher_is_better': True},
}
ALPHA = 0.05

## Table 1. Tasks, sizes, and cohort groups

This table documents which result file is used for each task. The sample size is the number of observations allocated to training **plus internal validation**, excluding held-out test data. Breast Cancer uses 455 and each Obesity task uses 1,688; the remaining tasks use 10,000. The two related pairs have distinct targets and are combined only when forming cohort-level summaries and tests.

In [ ]:
task_groups = {task: group for group, tasks in COHORT_GROUPS.items() for task in tasks}
if len(task_groups) != sum(map(len, COHORT_GROUPS.values())) or set(task_groups) != set(TASK_SIZES):
    raise ValueError('Cohort groups must contain every selected task exactly once.')
if len(MODELS) < 3 or len(set(MODELS)) != len(MODELS) or not METRICS:
    raise ValueError('Select at least three unique models and at least one metric.')
if not RESULTS_DIR.is_dir():
    raise FileNotFoundError(f'Results directory not found: {RESULTS_DIR}')

task_table = pd.DataFrame([
    {'Task': task, 'Train + validation N': size, 'Cohort group': task_groups[task]}
    for task, size in TASK_SIZES.items()
])
display(task_table)

## Table 2. Input coverage

Every selected model must have a saved result for every task at its specified size. This table identifies missing files or metrics. A result must include stored training indices, test indices and test labels; these are checked across models within the same task before any comparison. The notebook does not silently fill missing results or change the set of models between tasks.

In [ ]:
records, problems = [], []
reference_splits = {}
for task, size in TASK_SIZES.items():
    for model in MODELS:
        path = RESULTS_DIR / str(size) / task / f'{model}.pkl'
        if not path.is_file():
            problems.append({'Task': task, 'N': size, 'Model': model, 'Problem': 'Result file not found'})
            continue
        with path.open('rb') as file:
            result = pickle.load(file)
        missing_fields = [key for key in ('train_indices', 'test_indices', 'y_test', 'metric_mean') if key not in result]
        if missing_fields:
            problems.append({'Task': task, 'N': size, 'Model': model, 'Problem': f'Missing fields: {missing_fields}'})
            continue
        if result.get('dataset') != task or result.get('model') != model or result.get('train_size') != size:
            problems.append({'Task': task, 'N': size, 'Model': model, 'Problem': 'Result metadata does not match the requested combination'})
            continue
        split = tuple(np.asarray(result[key]).reshape(-1) for key in ('train_indices', 'test_indices', 'y_test'))
        if task in reference_splits:
            if not all(np.array_equal(left, right) for left, right in zip(reference_splits[task], split)):
                problems.append({'Task': task, 'N': size, 'Model': model, 'Problem': 'Training/test indices or test labels differ from the reference model'})
                continue
        else:
            reference_splits[task] = split
        for metric in METRICS:
            value = result['metric_mean'].get(metric)
            if value is None or not np.isfinite(value):
                problems.append({'Task': task, 'N': size, 'Model': model, 'Problem': f'Missing or non-finite metric: {metric}'})
                continue
            records.append({'task': task, 'size': size, 'group': task_groups[task], 'model': model, 'metric': metric, 'value': float(value)})

coverage = pd.DataFrame(problems, columns=['Task', 'N', 'Model', 'Problem'])
if coverage.empty:
    print(f'Complete coverage: {len(TASK_SIZES)} tasks × {len(MODELS)} models × {len(METRICS)} metrics.')
else:
    display(coverage)
    raise ValueError('Complete the missing or inconsistent results, or revise the selection, before continuing.')
scores = pd.DataFrame(records)

## Table 3. Cohort-level scores used by Wilcoxon and Friedman

For each metric and model, the score for a related pair is the arithmetic mean of its two task scores. The other groups retain their individual task score. Thus each model has four paired scores per metric. These values are used for both statistical tests.

This is an explicitly chosen summary across different objectives. In particular, binary F1 refers to the positive class while multiclass F1 is support-weighted, and the corresponding precision/recall definitions also differ. Arithmetic means across tasks should therefore be interpreted cautiously. The table remains descriptive even when scores share the same numeric scale.

In [ ]:
group_scores = (
    scores.groupby(['metric', 'group', 'model'], sort=False)['value']
    .mean()
    .unstack('model')
    .reindex(columns=MODELS)
)
expected_groups = pd.MultiIndex.from_product([list(METRICS), list(COHORT_GROUPS)], names=['metric', 'group'])
group_scores = group_scores.reindex(expected_groups)
if group_scores.isna().any().any():
    raise ValueError('A cohort/model/metric score is missing.')
display(group_scores.rename(columns=MODEL_LABELS).rename_axis(index=['Metric', 'Cohort group']).round(4))

## Table 4. Four reciprocal-rank contributions

For each metric, models are ranked **within each task** using unrounded saved scores. Better values receive rank 1; exact ties receive average ranks. The reciprocal of the task rank is computed first. The two reciprocal ranks for a related pair are then averaged within its cohort group. The table therefore shows the four inputs to each model's cohort-weighted MRR.

These are descriptive rankings, not statistical tests. They do not measure the magnitude or significance of a performance difference. The rankings here are formed within tasks; the tests below instead rank or compare the cohort-averaged metric scores shown in Table 3.

In [ ]:
task_ranks = scores.copy()
for metric, settings in METRICS.items():
    mask = task_ranks['metric'] == metric
    task_ranks.loc[mask, 'rank'] = (
        task_ranks.loc[mask].groupby('task')['value']
        .rank(method='average', ascending=not settings['higher_is_better'])
    )
task_ranks['reciprocal_rank'] = 1.0 / task_ranks['rank']
cohort_rr = (
    task_ranks.groupby(['metric', 'group', 'model'], sort=False)['reciprocal_rank']
    .mean()
    .unstack('model')
    .reindex(index=expected_groups, columns=MODELS)
)
display(cohort_rr.rename(columns=MODEL_LABELS).rename_axis(index=['Metric', 'Cohort group']).round(4))

## Table 5. Cohort-weighted MRR by metric and overall

Each metric's MRR is the mean of the four cohort-level reciprocal-rank contributions in Table 4. The **Overall** row is the equal-weight mean across the selected metrics, as in the earlier combined MRR. A larger MRR indicates more frequent high ranks and gives a large advantage to rank 1 over rank 2.

The overall row weights correlated metrics separately. For multiclass tasks, weighted recall equals accuracy; the two are nevertheless both included when selected here to retain the historical set of metrics. MRR is descriptive and depends on the selected models, tasks, metrics, and tie rule.

In [ ]:
mrr = cohort_rr.groupby(level='metric', sort=False).mean().reindex(METRICS)
mrr.index = [METRICS[metric]['label'] for metric in mrr.index]
mrr.loc['Overall'] = mrr.mean(axis=0)
display(mrr.rename(columns=MODEL_LABELS).rename_axis(index='Metric').round(4))

## Table 6. One-sided paired Wilcoxon comparisons across cohort groups

For each metric, the reference is selected as the model with the best arithmetic mean of the four cohort scores in Table 3; all other models are compared with it using a one-sided paired Wilcoxon signed-rank test in the direction “candidate is worse”. Models remain paired by cohort name. P-values are adjusted with Holm separately within each metric; the reference is not included in the correction.

**Exploratory limitation:** the reference is chosen from the same data, so these p-values and Holm adjustments do not fully correct for selecting the winner. Four cohort groups provide very low power. With four non-zero differences, the ideal minimum one-sided exact p-value is 0.0625 before correction; zeros and ties alter the null distribution. `*` means only that the corrected p-value is at least the chosen alpha; it does **not** establish equivalence. No marker is assigned to a missing test. The displayed p-values use SciPy's `method='auto'`, including its handling of zeros and ties. See [SciPy Wilcoxon documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.wilcoxon.html).

In [ ]:
wilcoxon_rows = []
for metric, settings in METRICS.items():
    matrix = group_scores.loc[metric].reindex(index=COHORT_GROUPS, columns=MODELS)
    direction = 1.0 if settings['higher_is_better'] else -1.0
    oriented = direction * matrix
    reference = oriented.mean(axis=0).idxmax()
    start = len(wilcoxon_rows)
    for model in MODELS:
        if model == reference:
            continue
        differences = (oriented[model] - oriented[reference]).to_numpy()
        n_nonzero = int(np.count_nonzero(differences))
        if n_nonzero == 0:
            statistic, p_value, test_note = 0.0, 1.0, 'All paired differences are zero'
        else:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always')
                test = stats.wilcoxon(differences, alternative='less', zero_method='wilcox', method='auto')
            statistic, p_value = float(test.statistic), float(test.pvalue)
            test_note = '; '.join(str(item.message) for item in caught)
        wilcoxon_rows.append({
            'Metric': settings['label'], 'Model': MODEL_LABELS[model],
            'Reference': MODEL_LABELS[reference], 'Cohorts': len(matrix),
            'Non-zero differences': n_nonzero,
            'Mean difference (candidate - reference)': float((matrix[model] - matrix[reference]).mean()),
            'W': statistic, 'p (raw)': p_value, 'Note': test_note,
        })
    metric_rows = wilcoxon_rows[start:]
    valid = [i for i, row in enumerate(metric_rows) if np.isfinite(row['p (raw)'])]
    if valid:
        adjusted = multipletests([metric_rows[i]['p (raw)'] for i in valid], alpha=ALPHA, method='holm')[1]
        for i, p_adjusted in zip(valid, adjusted):
            metric_rows[i]['p (Holm)'] = float(p_adjusted)
            metric_rows[i]['Not rejected'] = '*' if p_adjusted >= ALPHA else ''
wilcoxon_table = pd.DataFrame(wilcoxon_rows)
display(wilcoxon_table.round(4))

## Table 7. Friedman and Iman–Davenport tests

For each metric, the Friedman test compares all selected models on the four cohort-level scores from Table 3. SciPy computes the Friedman statistic with a correction for within-block ties. The Iman–Davenport transformation is

$$F_F = \frac{(N-1)\chi_F^2}{N(k-1)-\chi_F^2},$$

with degrees of freedom $k-1$ and $(k-1)(N-1)$, where $N$ is the number of cohort groups and $k$ is the number of models. These formulas correct the previous implementation's missing denominator term and incorrect degrees of freedom. [Demšar, 2006](https://www.jmlr.org/papers/volume7/demsar06a/demsar06a.pdf).

Both displayed p-values use asymptotic reference distributions and are **exploratory and unreliable with only four groups**. If all models tie within every group the test is undefined and reported as `NaN`. A significant omnibus result does not identify which pairs differ. No test is formed by treating different metrics or training sizes as independent datasets.

In [ ]:
friedman_rows = []
for metric, settings in METRICS.items():
    matrix = group_scores.loc[metric].reindex(index=COHORT_GROUPS, columns=MODELS)
    values = matrix.to_numpy(dtype=float)
    n_groups, n_models = values.shape
    if np.all(np.ptp(values, axis=1) == 0):
        chi_square, p_friedman = np.nan, np.nan
        f_statistic, p_iman = np.nan, np.nan
        note = 'All models tie within every cohort group; test undefined.'
    else:
        friedman = stats.friedmanchisquare(*[values[:, i] for i in range(n_models)])
        chi_square, p_friedman = float(friedman.statistic), float(friedman.pvalue)
        denominator = n_groups * (n_models - 1) - chi_square
        if np.isclose(denominator, 0.0, atol=1e-12):
            f_statistic, p_iman = np.inf, 0.0
        elif denominator < 0:
            raise ValueError(f'Invalid Iman–Davenport denominator for {metric}: {denominator}')
        else:
            f_statistic = (n_groups - 1) * chi_square / denominator
            p_iman = float(stats.f.sf(f_statistic, n_models - 1, (n_models - 1) * (n_groups - 1)))
        note = 'Asymptotic approximation; only four cohort groups.'
    friedman_rows.append({
        'Metric': settings['label'], 'Cohorts': n_groups, 'Models': n_models,
        'Friedman chi-square': chi_square, 'Friedman df': n_models - 1,
        'Friedman p': p_friedman, 'Iman–Davenport F': f_statistic,
        'F df1': n_models - 1, 'F df2': (n_models - 1) * (n_groups - 1),
        'Iman–Davenport p': p_iman, 'Note': note,
    })
friedman_table = pd.DataFrame(friedman_rows)
display(friedman_table.round(4))

## Interpretation

The six tasks are retained for within-task ranking, but the two shared-subject pairs each contribute a single averaged group to MRR and to the tests. The largest training sizes differ between tasks and should not be interpreted as a controlled comparison at a common N. Grouping does not prove independence between the resulting cohorts.

The statistical analyses are conditional on the saved trained models and selected test sets; they do not quantify variability across retraining or random splits. With four groups, the Wilcoxon test has a very coarse p-value resolution and the Friedman/Iman–Davenport asymptotic approximations are unreliable. A non-significant result is not evidence of equivalence. Report these tables as exploratory and retain the task-level metrics and confidence intervals alongside them.